In [1]:
import os

os.makedirs("/kaggle/working/read_prep", exist_ok=True)
print("Thư mục sẵn sàng:", os.listdir("/kaggle/working"))

Thư mục sẵn sàng: ['read_prep', '__notebook__.ipynb']


In [2]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset("yaful/MAGE")
df_mage_train = ds["train"].to_pandas()
df_mage_valid = ds["validation"].to_pandas()

print("Train shape:", df_mage_train.shape)
print("Unique labels:", df_mage_train["label"].unique())
print("Top src:\n", df_mage_train["src"].value_counts().head(15))

df_mage_train.to_parquet("/kaggle/working/read_prep/mage_train.parquet", index=False)
df_mage_valid.to_parquet("/kaggle/working/read_prep/mage_valid.parquet", index=False)

README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  404MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv: reconstructing file:   0%|          |  0.00B / 72.3MB            

valid.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 71.7MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/319071 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/56792 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60743 [00:00<?, ? examples/s]

Train shape: (319071, 3)
Unique labels: [1 0]
Top src:
 src
yelp_human                                 31827
eli5_human                                 16706
squad_human                                15820
wp_human                                    6356
xsum_human                                  4708
sci_gen_human                               4436
cmv_human                                   4223
roct_human                                  3287
hswag_human                                 3129
tldr_human                                  2826
xsum_machine_specified_gpt-3.5-trubo         800
wp_machine_topical_text-davinci-003          800
xsum_machine_continuation_t0_3b              800
xsum_machine_topical_gpt-3.5-trubo           800
xsum_machine_continuation_gpt-3.5-trubo      800
Name: count, dtype: int64


In [3]:
import glob
import subprocess

repo_dir = "/kaggle/working/read_prep/l2r_data"
if not os.path.exists(repo_dir):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ranhli/l2r_data.git", repo_dir],
        check=True,
    )

files = glob.glob(f"{repo_dir}/**/*", recursive=True)
data_files = [f for f in files if os.path.isfile(f)]
print("Tổng số file lấy được:", len(data_files))
print("Ví dụ 10 file đầu:", data_files[:10])

Cloning into '/kaggle/working/read_prep/l2r_data'...


Tổng số file lấy được: 107
Ví dụ 10 file đầu: ['/kaggle/working/read_prep/l2r_data/LICENSE.txt', '/kaggle/working/read_prep/l2r_data/README.md', '/kaggle/working/read_prep/l2r_data/TechnicalWriting/GPT-4o.json', '/kaggle/working/read_prep/l2r_data/TechnicalWriting/Llama-3-70B.json', '/kaggle/working/read_prep/l2r_data/TechnicalWriting/GPT-3-Turbo.json', '/kaggle/working/read_prep/l2r_data/TechnicalWriting/Gemini-1.5-Pro.json', '/kaggle/working/read_prep/l2r_data/TechnicalWriting/human.json', '/kaggle/working/read_prep/l2r_data/Entertainment/GPT-4o.json', '/kaggle/working/read_prep/l2r_data/Entertainment/Llama-3-70B.json', '/kaggle/working/read_prep/l2r_data/Entertainment/GPT-3-Turbo.json']


In [4]:
import json
import glob
import os
import pandas as pd

# 1. Đọc lại MAGE và kiểm tra quy ước nhãn 0 / 1
df_mage = pd.read_parquet("/kaggle/working/read_prep/mage_train.parquet")
print("Kiểm tra nhãn trên MAGE:")
print(df_mage.groupby(["label"])["src"].apply(lambda s: list(s.unique()[:5])))

# Lọc 10 miền của MAGE theo pipeline READ:
# CMV, ELI5, HellaSwag (hswag), ROC (roct), SCI (sci_gen), SQuAD, TLDR, WP, XSum, Yelp
mage_prefixes = ("cmv_", "eli5_", "hswag_", "roct_", "sci_gen_", "squad_", "tldr_", "wp_", "xsum_", "yelp_")
df_mage_10 = df_mage[df_mage["src"].str.startswith(mage_prefixes)].copy()
print("Số dòng MAGE sau khi lọc 10 miền:", len(df_mage_10))

# 2. Kiểm tra cấu trúc 1 file JSON trong l2r_data
sample_json_path = "/kaggle/working/read_prep/l2r_data/Environmental/human.json"
with open(sample_json_path, "r", encoding="utf-8") as f:
    sample_data = json.load(f)

print("Kiểu dữ liệu file l2r_data:", type(sample_data))
if isinstance(sample_data, list):
    print("Số phần tử:", len(sample_data), "| Keys mẫu đầu:", sample_data[0].keys() if isinstance(sample_data[0], dict) else type(sample_data[0]))
elif isinstance(sample_data, dict):
    print("Keys gốc:", list(sample_data.keys())[:10])

Kiểm tra nhãn trên MAGE:
label
0    [cmv_machine_continuation_t0_11b, cmv_machine_...
1    [cmv_human, eli5_human, tldr_human, xsum_human...
Name: src, dtype: object
Số dòng MAGE sau khi lọc 10 miền: 319071
Kiểu dữ liệu file l2r_data: <class 'list'>
Số phần tử: 200 | Keys mẫu đầu: <class 'str'>


In [5]:
import glob
import json
import os
import pandas as pd

# 1. Chuẩn hóa MAGE (lưu ý: trong MAGE, label == 1 là Human, label == 0 là AI)
domain_map = {
    "cmv": "CMV",
    "eli5": "ELI5",
    "hswag": "HELLASWAG",
    "roct": "ROC",
    "sci_gen": "SCI",
    "squad": "SQUAD",
    "tldr": "TLDR",
    "wp": "WP",
    "xsum": "XSUM",
    "yelp": "YELP",
}

def extract_mage_domain(src_val):
    for prefix, dom in domain_map.items():
        if src_val.startswith(prefix + "_"):
            return dom
    return "OTHER"

df_mage_clean = pd.DataFrame({
    "text": df_mage_10["text"].astype(str).str.strip(),
    "label": df_mage_10["label"].map({1: "Human", 0: "AI"}),
    "domain": df_mage_10["src"].apply(extract_mage_domain),
    "sub_source": df_mage_10["src"],
    "source_dataset": "MAGE",
})

# 2. Đọc & chuẩn hóa toàn bộ 21 miền của l2r_data (Rewrite)
rewrite_rows = []
json_files = sorted(glob.glob("/kaggle/working/read_prep/l2r_data/*/*.json"))

for fp in json_files:
    sub_domain = os.path.basename(os.path.dirname(fp))
    file_stem = os.path.splitext(os.path.basename(fp))[0]  # vd: human, GPT-4o, Gemini-1.5-Pro...
    label_str = "Human" if file_stem.lower() == "human" else "AI"
    
    with open(fp, "r", encoding="utf-8") as f:
        items = json.load(f)
        
    for txt in items:
        if isinstance(txt, str) and txt.strip():
            rewrite_rows.append({
                "text": txt.strip(),
                "label": label_str,
                "domain": "REWRITE",
                "sub_source": f"{sub_domain}_{file_stem}",
                "source_dataset": "Rewrite",
            })

df_rewrite_clean = pd.DataFrame(rewrite_rows)

# 3. Gộp chung, lọc trùng và lưu Parquet
df_pool = pd.concat([df_mage_clean, df_rewrite_clean], ignore_index=True)
df_pool = df_pool[df_pool["text"].str.len() > 0].drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Tổng mẫu sau khi gộp & lọc trùng:", len(df_pool))
print("\nPhân bố theo source_dataset & label:")
print(pd.crosstab(df_pool["source_dataset"], df_pool["label"], margins=True))
print("\nPhân bố theo domain & label:")
print(pd.crosstab(df_pool["domain"], df_pool["label"], margins=True))

out_path = "/kaggle/working/read_prep/read_unified_pool.parquet"
df_pool.to_parquet(out_path, index=False)
print("\nĐã lưu thành công:", out_path)

Tổng mẫu sau khi gộp & lọc trùng: 338666

Phân bố theo source_dataset & label:
label               AI  Human     All
source_dataset                       
MAGE            225753  93318  319071
Rewrite          15681   3914   19595
All             241434  97232  338666

Phân bố theo domain & label:
label          AI  Human     All
domain                          
CMV         20388   4223   24611
ELI5        25548  16706   42254
HELLASWAG   24482   3129   27611
REWRITE     15681   3914   19595
ROC         25510   3287   28797
SCI         18691   4436   23127
SQUAD       19940  15820   35760
TLDR        19811   2826   22637
WP          24803   6356   31159
XSUM        26051   4708   30759
YELP        20529  31827   52356
All        241434  97232  338666

Đã lưu thành công: /kaggle/working/read_prep/read_unified_pool.parquet
